# Gen Data Sample Check
Load processed generation task data, extract 3 sample sets (interval=5000, 5 results each), display contents, and validate `ud_idx` uniqueness.

## 1. Import Libraries & Setup Paths

In [1]:
import os
import sys
import pickle
from collections import Counter, defaultdict
import pandas as pd

sw = 3

# ── Paths ──────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath(".")
ROOT_DIR     = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
DATA_DIR     = os.path.join(ROOT_DIR, "data", "gen_task")
PKL_PATH     = os.path.join(DATA_DIR, f"processed_gen_SlideWindow_{sw}.pkl")

# Fallback: try the older filename if the slide-window variant doesn't exist
if not os.path.exists(PKL_PATH):
    FALLBACK    = os.path.join(DATA_DIR, "processed_gen_data.pkl")
    if os.path.exists(FALLBACK):
        print(f"[WARN] Slide-window file not found; falling back to: {FALLBACK}")
        PKL_PATH = FALLBACK
    else:
        raise FileNotFoundError(
            f"No processed gen-data pkl found.\n"
            f"Tried:\n  {os.path.join(DATA_DIR, f'processed_gen_SlideWindow_{sw}.pkl')}\n  {FALLBACK}"
        )

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"DATA_DIR : {DATA_DIR}")
print(f"PKL_PATH : {PKL_PATH}")


ROOT_DIR : /home/new/Documents/QSH/Llama_train
DATA_DIR : /home/new/Documents/QSH/Llama_train/data/gen_task
PKL_PATH : /home/new/Documents/QSH/Llama_train/data/gen_task/processed_gen_SlideWindow_3.pkl


## 2. Load Processed `.pkl` Data

In [2]:
with open(PKL_PATH, "rb") as f:
    data_dict, max_len = pickle.load(f)

print("── Top-level keys ────────────────────────────────────")
print(list(data_dict.keys()))

total_samples = len(data_dict.get("emotion", data_dict.get("ud_idx", [])))
print(f"\n── Total samples      : {total_samples:,}")
print(f"── Max input len (≈)  : {max_len:,}")
print("\n── Per-key sample counts ─────────────────────────────")
for k, v in data_dict.items():
    print(f"  {k:20s} : {len(v):,}")


── Top-level keys ────────────────────────────────────
['input_text', 'emotion', 'ud_idx', 'ld_idx']

── Total samples      : 51,204
── Max input len (≈)  : 1,767

── Per-key sample counts ─────────────────────────────
  input_text           : 51,204
  emotion              : 51,204
  ud_idx               : 51,204
  ld_idx               : 51,204


## 3. Extract 3 Sample Sets (5 Results Each, Interval = 5000)

In [3]:
INTERVAL   = 5000
N_SETS     = 3
N_PER_SET  = 5

def slice_set(data: dict, start: int, n: int) -> dict:
    """Return a sub-dict with n entries starting from `start`."""
    end = start + n
    return {k: v[start:end] for k, v in data.items()}

sample_sets = {}
for s in range(N_SETS):
    start_idx = s * INTERVAL
    sample_sets[f"set_{s}"] = slice_set(data_dict, start_idx, N_PER_SET)
    print(f"set_{s}  →  indices [{start_idx} : {start_idx + N_PER_SET}]  "
          f"| ud_idx = {sample_sets[f'set_{s}']['ud_idx']}")


set_0  →  indices [0 : 5]  | ud_idx = [0, 0, 0, 1, 1]
set_1  →  indices [5000 : 5005]  | ud_idx = [2409, 2410, 2410, 2411, 2411]
set_2  →  indices [10000 : 10005]  | ud_idx = [4834, 4834, 4835, 4835, 4836]


## 4. Display Sample Set Contents

In [4]:
def display_message_list(messages):
    """Pretty-print a list of role/content dicts."""
    if not isinstance(messages, list):
        print(f"  [raw] {messages}")
        return
    for msg in messages:
        role    = msg.get("role", "?").upper()
        content = msg.get("content", "")
        # Truncate very long content for readability
        preview = content[:300] + " …" if len(content) > 300 else content
        print(f"  [{role}] {preview}")

SEP_MAJOR = "=" * 70
SEP_MINOR = "-" * 50

for set_name, sset in sample_sets.items():
    print(SEP_MAJOR)
    print(f"  SAMPLE SET: {set_name.upper()}")
    print(SEP_MAJOR)

    n = len(sset["ud_idx"])
    for i in range(n):
        print(f"\n  Result {i+1}/{n}")
        print(f"  emotion  : {sset['emotion'][i]}")
        print(f"  ud_idx   : {sset['ud_idx'][i]}")
        print(f"  ld_idx   : {sset['ld_idx'][i]}")
        print(f"  Messages :")
        display_message_list(sset["input_text"][i])
        if "target_text" in sset:
            tgt = sset["target_text"][i]
            preview = tgt[:200] + " …" if len(tgt) > 200 else tgt
            print(f"  target   : {preview}")
        print(SEP_MINOR)
    print()


  SAMPLE SET: SET_0

  Result 1/5
  emotion  : sentimental
  ud_idx   : 0
  ld_idx   : 0
  Messages :
  [SYSTEM] You are an empathetic assistant. Your task is to understand the user's emotion and feelings, and respond supportively. <background_info><situation>i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .</situation></background_info> …
  [USER] i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .
  [ASSISTANT] was this a friend you were in love with , or just a best friend ?
--------------------------------------------------

  Result 2/5
  emotion  : sentimental
  ud_idx   : 0
  ld_idx   : 1
  Messages :
  [SYSTEM] You are an empathetic assistant. Your task is to understand the user's emotion and feelings, and respond supportively. <background_info><situation>i rem

## 5. Unique Dialogue Index — Collision & Replication Check

In [5]:
ud_idx_all = data_dict["ud_idx"]

total_count  = len(ud_idx_all)
unique_count = len(set(ud_idx_all))
counter      = Counter(ud_idx_all)
duplicates   = {k: v for k, v in counter.items() if v > 1}

print("── ud_idx collision / replication report ─────────────────────────────")
print(f"  Total samples       : {total_count:,}")
print(f"  Unique ud_idx vals  : {unique_count:,}")
print(f"  Duplicate ud_idx    : {len(duplicates):,}")
print()

if duplicates:
    print("  [WARNING] Collisions detected! Top duplicates:")
    dup_df = (
        pd.DataFrame(
            [(k, v) for k, v in duplicates.items()],
            columns=["ud_idx", "occurrences"]
        )
        .sort_values("occurrences", ascending=False)
        .reset_index(drop=True)
    )
    print(dup_df.head(20).to_string(index=False))

    # Check if duplicates are intentional (same dialogue, different local turns)
    print("\n  Checking same ud_idx with different ld_idx …")
    from itertools import groupby

    indices_by_ud = defaultdict(list)
    for i, uid in enumerate(ud_idx_all):
        indices_by_ud[uid].append(i)

    mixed_ld = 0
    for uid, positions in indices_by_ud.items():
        if len(positions) > 1:
            ld_vals = [data_dict["ld_idx"][p] for p in positions]
            if len(set(ld_vals)) == len(ld_vals):
                # All ld_idx differ → expected multi-turn sliding window
                pass
            else:
                mixed_ld += 1

    if mixed_ld == 0:
        print("  ✓ All duplicate ud_idx entries have distinct ld_idx values → "
              "multi-turn sliding window behaviour (expected).")
    else:
        print(f"  ✗ {mixed_ld} ud_idx values share both identical ud_idx AND ld_idx "
              "→ TRUE collision!")
else:
    print("  ✓ No collisions — every ud_idx is unique across all samples.")

print()
print("── Sanity: value range of ud_idx ─────────────────────────────────────")
print(f"  min={min(ud_idx_all)}, max={max(ud_idx_all)}, "
      f"span={max(ud_idx_all)-min(ud_idx_all)+1}")


── ud_idx collision / replication report ─────────────────────────────
  Total samples       : 51,204
  Unique ud_idx vals  : 24,795
  Duplicate ud_idx    : 24,795

  [WARNING] Collisions detected! Top duplicates:
 ud_idx  occurrences
  11609            4
  22366            4
  18025            4
  18024            4
   6444            4
  18021            4
    620            4
  13202            4
  18020            4
   1250            4
  18018            4
  17801            4
  20645            4
  18016            4
  22081            4
   4994            4
  11061            4
  13737            4
  17806            4
   3476            4

  Checking same ud_idx with different ld_idx …
  ✓ All duplicate ud_idx entries have distinct ld_idx values → multi-turn sliding window behaviour (expected).

── Sanity: value range of ud_idx ─────────────────────────────────────
  min=0, max=24794, span=24795


In [6]:
# ── Optional: quick DataFrame overview of all sample-set ud_idx ──────────────
rows = []
for set_name, sset in sample_sets.items():
    for i in range(len(sset["ud_idx"])):
        rows.append({
            "set":     set_name,
            "result":  i,
            "ud_idx":  sset["ud_idx"][i],
            "ld_idx":  sset["ld_idx"][i],
            "emotion": sset["emotion"][i],
        })

overview_df = pd.DataFrame(rows)
print(overview_df.to_string(index=False))

# Cross-set ud_idx collision check
cross_dupes = Counter(overview_df["ud_idx"])
cross_dupes = {k: v for k, v in cross_dupes.items() if v > 1}
if cross_dupes:
    print(f"\n[WARNING] Cross-set ud_idx collisions: {cross_dupes}")
else:
    print("\n✓ No cross-set ud_idx collisions among the 3 sample sets.")


  set  result  ud_idx  ld_idx      emotion
set_0       0       0       0  sentimental
set_0       1       0       1  sentimental
set_0       2       0       2  sentimental
set_0       3       1       0       afraid
set_0       4       1       1       afraid
set_1       0    2409       1       joyful
set_1       1    2410       0      annoyed
set_1       2    2410       1      annoyed
set_1       3    2411       0    surprised
set_1       4    2411       1    surprised
set_2       0    4834       0    impressed
set_2       1    4834       1    impressed
set_2       2    4835       0    disgusted
set_2       3    4835       1    disgusted
set_2       4    4836       0 disappointed

[WARNING] Cross-set ud_idx collisions: {0: 3, 1: 2, 2410: 2, 2411: 2, 4834: 2, 4835: 2}
